# LoRA fine-tuning for grounded RAG (Kaggle 2x T4)
Settings: **Accelerator = GPU T4 x2**, **Internet = On**. Add your bundle as a Dataset (Kaggle auto-extracts zips).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U trl peft bitsandbytes accelerate datasets
import transformers, trl, peft, bitsandbytes, accelerate
print({m.__name__: m.__version__ for m in (transformers, trl, peft, bitsandbytes, accelerate)})


## Copy the project from the Kaggle dataset into /kaggle/working

In [ ]:
import os, glob, shutil, zipfile
os.chdir("/kaggle/working")
found = glob.glob("/kaggle/input/**/train.py", recursive=True)
if found:  # dataset was auto-extracted
    src = os.path.dirname(found[0])
    print("Copying from", src)
    for name in os.listdir(src):
        s, d = os.path.join(src, name), os.path.join(".", name)
        if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
        else: shutil.copy2(s, d)
else:  # dataset still contains a zip
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("*.zip")
    assert zips, "No train.py or zip found under /kaggle/input - check the dataset is attached"
    print("Extracting", zips[0]); zipfile.ZipFile(zips[0]).extractall(".")
assert os.path.exists("train.py") and os.path.exists("eval.py"), "train.py / eval.py missing"
!ls data/qasper_generated data/arxiv_test && wc -l data/qasper_generated/*.jsonl data/arxiv_test/test*.jsonl

## Pre-download the model (avoids both GPU processes downloading at once)

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-3B-Instruct")

## Sanity check (about 10 steps). Must print `[PASS]` for masking and finish eval + checkpoint save

In [ ]:
!accelerate launch --multi_gpu --num_processes=2 --mixed_precision fp16 train.py \
    --model-name Qwen/Qwen2.5-3B-Instruct \
    --output-dir results/sanity --sanity-check \
    --batch-size 1 --grad-accum 4

## Full training (DDP on both T4s)
Effective batch = 2 GPUs x 1 x 4 = 8, about 188 steps per epoch. Run time is not measured yet: read the step speed from the sanity run and extrapolate before trusting any estimate.

In [ ]:
!accelerate launch --multi_gpu --num_processes=2 --mixed_precision fp16 train.py \
    --model-name Qwen/Qwen2.5-3B-Instruct \
    --train-file data/qasper_generated/train.jsonl \
    --val-file data/qasper_generated/val.jsonl \
    --output-dir results/qwen-3b-lora \
    --epochs 3 --batch-size 1 --grad-accum 4 --lr 2e-4

## Evaluate: base model and LoRA on the same test set (single process)

In [ ]:
!python eval.py --model-name Qwen/Qwen2.5-3B-Instruct \
    --output-file results/base_eval_report.json

In [ ]:
!python eval.py --model-name Qwen/Qwen2.5-3B-Instruct \
    --adapter-path results/qwen-3b-lora \
    --output-file results/lora_eval_report.json

## Package the adapter

In [ ]:
!zip -rq qwen2.5_3b_rag_lora_weights.zip results/qwen-3b-lora results/*_eval_report.json
print("Download qwen2.5_3b_rag_lora_weights.zip from the Output panel")